<a href="https://colab.research.google.com/github/Sai875/flyrank-ml-internship/blob/main/w04_baseline_score.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-07 — Baseline Action Score and Top-10 Review

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 0. Setup

Loading the starter dataset (`data/raw/content_refresh_anonymized.csv`, Lane 2's smaller-but-complete slice — every column this baseline needs is already in it, so no warehouse round trip needed for a rule this simple). `is_declining_label` is rebuilt here **only as an evaluation label** for the signal checks and precision@K below — it is never a feature, and `trend_direction` / `trend_pct` never enter the score.

In [10]:
import os, sys, subprocess
from pathlib import Path

REPO_URL = "https://github.com/pretom26/ml_internship_flyrankAI.git"
DATA_RELATIVE_PATH = Path("data/raw/content_refresh_anonymized.csv")

# Work from an existing repository when possible; otherwise clone it for a fresh notebook session.
repo_root = None
for candidate in [Path.cwd(), *Path.cwd().parents]:
    if (candidate / DATA_RELATIVE_PATH).exists():
        repo_root = candidate
        break

if repo_root is None:
    repo_dir = Path("/content/ml_internship_flyrankAI") if "google.colab" in sys.modules else Path.cwd() / "ml_internship_flyrankAI"
    if not repo_dir.exists():
        subprocess.run(["git", "clone", "--depth", "1", REPO_URL, str(repo_dir)], check=True)
    repo_root = repo_dir

os.chdir(repo_root)

import pandas as pd
import numpy as np

data_path = repo_root / DATA_RELATIVE_PATH
df = pd.read_csv(data_path)

required_columns = {
    "trend_direction", "days_since_last_update", "position_tier",
    "ctr", "impressions_90d", "content_id", "client_id",
    "avg_position"
}
missing_columns = required_columns.difference(df.columns)
if missing_columns:
    raise ValueError(f"Dataset is missing required columns: {sorted(missing_columns)}")

# Evaluation label only. It is never included in the action score.
df["is_declining_label"] = df["trend_direction"].astype("string").str.lower().eq("down").astype(int)

print(f"{df.shape[0]:,} pages | overall decline rate: {df['is_declining_label'].mean():.3f}")
print(f"Dataset: {data_path}")


30,000 pages | overall decline rate: 0.542
Dataset: /content/ml_internship_flyrankAI/data/raw/content_refresh_anonymized.csv


## 1. My rule and its reason codes

**Plan.** Two signals get checked against the evaluation label before anything is coded. Both are the signal *behind* a real FlyRank flag from the session — I'm not inventing new ones:

1. **Staleness** — the signal behind the refresh flags. Claim: *pages that haven't been updated in a while decline more often.*
2. **CTR vs. position** — the signal behind the CTR-fix logic. Claim: *pages clicking worse than peers at the same search position decline more often.*

Each gets one bucket table with n, and a one-word verdict: CONFIRMED / OPPOSITE / MIXED / FALSE.

### Signal check 1 — Staleness (behind the refresh flags)

`days_since_last_update` is heavy-tailed with a hard step in the data around 90 days (median is 20, but a large block of pages sits at 90+). Bucketing at that natural step — rather than the 180-day number I first guessed — is what the data actually supports.

In [11]:
df["staleness_bucket"] = np.where(
    pd.to_numeric(df["days_since_last_update"], errors="coerce") >= 90,
    "stale_90d_plus",
    "updated_lt_90d",
)

staleness_table = (
    df.groupby("staleness_bucket", dropna=False)["is_declining_label"]
      .agg(n="count", decline_rate="mean")
      .reindex(["updated_lt_90d", "stale_90d_plus"])
)
display(staleness_table)


,n,decline_rate
staleness_bucket,,
updated_lt_90d,20655,0.512031
stale_90d_plus,9345,0.608454


**Interpretation.** The next signal-summary cell calculates the verdict from these observed rates each time you run the notebook. A higher decline rate in `stale_90d_plus` supports the signal directionally; a lower rate points the other way. This is descriptive evidence, not proof that staleness causes decline.


### Signal check 2 — CTR vs. position (behind the CTR-fix logic)

A flat CTR threshold isn't fair across positions — position 3 and position 30 have very different "normal" CTRs. So "underperforming" is defined *relative to peers in the same `position_tier`*: below that tier's median CTR. Rows with no position data are excluded (they can't be compared).

In [12]:
valid = df.loc[df["position_tier"].notna() & (df["position_tier"].astype(str) != "no_data")].copy()
valid["ctr"] = pd.to_numeric(valid["ctr"], errors="coerce")
expected_ctr_by_tier = valid.groupby("position_tier")["ctr"].transform("median")
valid = valid.loc[valid["ctr"].notna() & expected_ctr_by_tier.notna()].copy()
expected_ctr_by_tier = valid.groupby("position_tier")["ctr"].transform("median")
valid["ctr_bucket"] = np.where(
    valid["ctr"] < expected_ctr_by_tier,
    "underperform_ctr",
    "at_or_above_tier_median",
)

ctr_table = (
    valid.groupby("ctr_bucket")["is_declining_label"]
         .agg(n="count", decline_rate="mean")
         .reindex(["at_or_above_tier_median", "underperform_ctr"])
)
display(ctr_table)


,n,decline_rate
ctr_bucket,,
at_or_above_tier_median,16921,0.502630
underperform_ctr,13079,0.593088


In [13]:
def signal_verdict(table, higher_risk_bucket, comparison_bucket, min_n=1, tolerance=1e-12):
    """Return a data-driven directional verdict; unavailable/small groups are not called confirmed."""
    if higher_risk_bucket not in table.index or comparison_bucket not in table.index:
        return "FALSE (comparison unavailable)"
    risk_row, comparison_row = table.loc[higher_risk_bucket], table.loc[comparison_bucket]
    if pd.isna(risk_row["decline_rate"]) or pd.isna(comparison_row["decline_rate"]):
        return "FALSE (comparison unavailable)"
    if min(int(risk_row["n"]), int(comparison_row["n"])) < min_n:
        return "FALSE (insufficient data)"
    difference = float(risk_row["decline_rate"] - comparison_row["decline_rate"])
    if difference > tolerance:
        return "CONFIRMED (directional)"
    if difference < -tolerance:
        return "OPPOSITE"
    return "MIXED (rates effectively equal)"

staleness_verdict = signal_verdict(
    staleness_table, "stale_90d_plus", "updated_lt_90d"
)
ctr_verdict = signal_verdict(
    ctr_table, "underperform_ctr", "at_or_above_tier_median"
)

signal_summary = pd.DataFrame([
    {"signal": "Staleness (90+ days)", "verdict": staleness_verdict,
     "risk_bucket_rate": staleness_table.loc["stale_90d_plus", "decline_rate"],
     "comparison_rate": staleness_table.loc["updated_lt_90d", "decline_rate"]},
    {"signal": "CTR below position-tier median", "verdict": ctr_verdict,
     "risk_bucket_rate": ctr_table.loc["underperform_ctr", "decline_rate"],
     "comparison_rate": ctr_table.loc["at_or_above_tier_median", "decline_rate"]},
])
display(signal_summary)
print("Signal verdicts are computed from the current dataset; they are not hard-coded.")


,signal,verdict,risk_bucket_rate,comparison_rate
0,Staleness (90+ days),CONFIRMED (directional),0.608454,0.512031
1,CTR below position-tier median,CONFIRMED (directional),0.593088,0.502630


Signal verdicts are computed from the current dataset; they are not hard-coded.


**Interpretation.** The signal-summary cell calculates whether `underperform_ctr` has a higher decline rate than pages at or above their position-tier median. Rows with missing CTR or no comparable position tier are excluded. This is descriptive evidence, not proof that low CTR causes decline.


### The rule, in plain words

> A page is worth reviewing first if it's **stale** (no update in 90+ days), still **visible** (≥500 impressions in 90 days — so there's an audience to lose), and its **click-through rate is underperforming** its own position tier. Rank the pages that clear all three bars by how much exposure (impressions) is riding on them.

The two signal verdicts are printed in the **Signal summary** table above. The rule is intentionally evaluated against those observed results rather than assuming either signal is confirmed in advance. If one or both signals are not directionally supported, treat this rule as a candidate baseline to compare—not as a validated causal explanation.

**Reason code (one, fixed):** `stale_visible_ctr_underperform` — every flagged row scores for the same reason, by design; that's what makes a hand rule readable in one sentence. Rows that don't clear all three bars get `no_flag`.

**Action label:** `refresh_review` for flagged rows, `monitor` otherwise.


## 2. Build the ranked queue (writes the CSV)

In [14]:
# Convert scoring inputs explicitly so missing or malformed values do not silently score as eligible.
days_since_update = pd.to_numeric(df["days_since_last_update"], errors="coerce")
impressions = pd.to_numeric(df["impressions_90d"], errors="coerce").fillna(0)
ctr_values = pd.to_numeric(df["ctr"], errors="coerce")

stale = (days_since_update >= 90).fillna(False).astype(int)
visible = (impressions >= 500).astype(int)

has_position_tier = df["position_tier"].notna() & (df["position_tier"].astype(str) != "no_data")
score_frame = df.loc[has_position_tier].copy()
score_frame["_ctr_numeric"] = ctr_values.loc[has_position_tier]
pos_median_ctr = score_frame.groupby("position_tier")["_ctr_numeric"].median()
expected_ctr = df["position_tier"].map(pos_median_ctr)
ctr_underperform = (
    has_position_tier & ctr_values.notna() & expected_ctr.notna() & (ctr_values < expected_ctr)
).astype(int)

# The label is evaluation-only and never participates in this score.
df["action_score"] = stale * visible * ctr_underperform * impressions
df["reason_code"] = np.where(df["action_score"] > 0, "stale_visible_ctr_underperform", "no_flag")
df["action_label"] = np.where(df["action_score"] > 0, "refresh_review", "monitor")

n_flagged = int((df["action_score"] > 0).sum())
print(f"pages flagged for review: {n_flagged:,} of {len(df):,}")

# Stable sorting makes tied scores reproducible in this dataset's existing row order.
def precision_at_k(scores, labels, k):
    scores = np.asarray(scores, dtype=float)
    labels = np.asarray(labels, dtype=int)
    if len(scores) != len(labels):
        raise ValueError("scores and labels must have the same length")
    if k <= 0 or len(scores) == 0:
        return float("nan")
    k_eff = min(int(k), len(scores))
    order = np.argsort(-np.nan_to_num(scores, nan=-np.inf), kind="stable")
    return float(labels[order[:k_eff]].mean())

y = df["is_declining_label"].to_numpy(dtype=int)
base_rate = float(y.mean()) if len(y) else float("nan")
print(f"base rate (declining, unranked): {base_rate:.3f}")

# Standard Precision@K over all rows is shown for comparison, but it can include monitor rows
# when fewer than K pages are flagged. Also report precision within the actionable queue.
flagged_mask = df["action_score"].to_numpy() > 0
flagged_scores = df.loc[flagged_mask, "action_score"].to_numpy()
flagged_labels = df.loc[flagged_mask, "is_declining_label"].to_numpy(dtype=int)
for k in (10, 20, 50, 100):
    all_p = precision_at_k(df["action_score"], y, k)
    if n_flagged:
        queue_p = precision_at_k(flagged_scores, flagged_labels, k)
        actual_k = min(k, n_flagged)
        print(f"Precision@{k} (all rows; effective K={min(k, len(df))}): {all_p:.3f} | flagged queue P@{k} (effective K={actual_k}): {queue_p:.3f}")
    else:
        print(f"Precision@{k} (all rows; effective K={min(k, len(df))}): {all_p:.3f} | flagged queue P@{k}: n/a (no pages flagged)")


pages flagged for review: 2,019 of 30,000
base rate (declining, unranked): 0.542
Precision@10 (all rows; effective K=10): 0.700 | flagged queue P@10 (effective K=10): 0.700
Precision@20 (all rows; effective K=20): 0.650 | flagged queue P@20 (effective K=20): 0.650
Precision@50 (all rows; effective K=50): 0.520 | flagged queue P@50 (effective K=50): 0.520
Precision@100 (all rows; effective K=100): 0.530 | flagged queue P@100 (effective K=100): 0.530


In [15]:
ranked = df.sort_values("action_score", ascending=False).reset_index(drop=True)
ranked.insert(0, "rank", ranked.index + 1)

out_cols = ["rank", "content_id", "client_id", "action_score", "reason_code", "action_label",
            "impressions_90d", "days_since_last_update", "avg_position", "position_tier", "ctr",
            "trend_direction"]

out_path = Path("work/outputs/baseline_action_score.csv")
out_path.parent.mkdir(parents=True, exist_ok=True)
ranked[out_cols].to_csv(out_path, index=False)
print(f"wrote {len(ranked):,} rows to {out_path}")

wrote 30,000 rows to work/outputs/baseline_action_score.csv


**Reading precision@K.** The output reports standard Precision@K over all rows and Precision@K within the actionable (`action_score > 0`) queue. Each metric prints its effective K, because K is capped at the number of available rows. If fewer than K pages are flagged, the all-row metric includes `monitor` rows with zero scores; use the flagged-queue metric to judge the precision of the actual review queue. Tied scores are sorted stably so the output is reproducible for a fixed dataset order. These are descriptive baseline metrics, not proof of causation.


## 3. Top-10 review

In [16]:
review_cols = ["rank", "content_id", "client_id", "impressions_90d", "days_since_last_update",
               "avg_position", "position_tier", "ctr", "trend_direction", "action_label"]
ranked[review_cols].head(10)

,rank,content_id,client_id,impressions_90d,days_since_last_update,avg_position,position_tier,ctr,trend_direction,action_label
0,1,content_5fe46e04994d,client_4e07408562,517715,104,4.2,page_1,0.14,down,refresh_review
1,2,content_36ff89c8214e,client_19581e27de,295097,104,7.3,page_1,0.05,stable,refresh_review
2,3,content_c8e9d6ab9013,client_19581e27de,208678,104,9.7,page_1,0.00,down,refresh_review
3,4,content_a7427266c305,client_19581e27de,201111,104,5.7,page_1,0.11,stable,refresh_review
4,5,content_91652435f57a,client_19581e27de,159590,104,7.8,page_1,0.06,stable,refresh_review
5,6,content_f42eb861c6dd,client_19581e27de,152467,104,6.5,page_1,0.13,down,refresh_review
6,7,content_11fcfd65d94c,client_19581e27de,149083,104,6.2,page_1,0.15,down,refresh_review
7,8,content_97a86caf3a3d,client_19581e27de,147670,104,6.4,page_1,0.07,down,refresh_review
8,9,content_8b36799b7e44,client_6208ef0f77,141400,104,32.0,page_3_5,0.02,down,refresh_review
9,10,content_c1fe78bc4e37,client_19581e27de,134055,104,7.5,page_1,0.03,down,refresh_review


The code cell above displays the current top 10 from the dataset. Review those rows directly rather than relying on a fixed list of IDs or values from an earlier run. For each candidate, check whether the CTR gap is meaningful, whether zero CTR reflects missing click data, whether the page's search intent is appropriate, and whether a refresh has already happened since the snapshot.

## 4. Weak picks + leakage check

In [17]:
# Concentration check: is the top of the queue dominated by a couple of clients?
print("client_id counts in top 20:")
print(ranked.head(20)["client_id"].value_counts())
print()
print("days_since_last_update values in top 20:")
print(ranked.head(20)["days_since_last_update"].value_counts())
print()
print("trend_direction mix in top 20 (label agreement, not used by the rule):")
print(ranked.head(20)["trend_direction"].value_counts())

client_id counts in top 20:
client_id
client_19581e27de    14
client_6208ef0f77     5
client_4e07408562     1
Name: count, dtype: int64

days_since_last_update values in top 20:
days_since_last_update
104    20
Name: count, dtype: int64

trend_direction mix in top 20 (label agreement, not used by the rule):
trend_direction
down      13
stable     4
up         3
Name: count, dtype: int64


In [18]:
# Segment analysis is limited to tiers with at least one flagged candidate.
segment_results = []
for tier in sorted(df.loc[df["action_score"] > 0, "position_tier"].dropna().unique()):
    tier_df = df.loc[df["position_tier"] == tier].copy()
    y_tier = tier_df["is_declining_label"].to_numpy(dtype=int)
    base_tier = float(y_tier.mean()) if len(y_tier) else float("nan")
    tier_flagged = tier_df.loc[tier_df["action_score"] > 0]
    n_tier_flagged = len(tier_flagged)
    # P@20 within the actionable candidates for this tier; effective K is capped at available flags.
    p20 = precision_at_k(
        tier_flagged["action_score"].to_numpy(),
        tier_flagged["is_declining_label"].to_numpy(dtype=int),
        20,
    ) if n_tier_flagged else float("nan")
    segment_results.append({
        "tier": tier,
        "n_pages": len(tier_df),
        "n_flagged": n_tier_flagged,
        "base_decline_rate": base_tier,
        "flagged_queue_p_at_20": p20,
        "lift_vs_tier_base": p20 - base_tier if n_tier_flagged else float("nan"),
    })

segment_df = pd.DataFrame(segment_results)
if not segment_df.empty:
    segment_df = segment_df.sort_values("lift_vs_tier_base", ascending=False, na_position="last")
display(segment_df)


,tier,n_pages,n_flagged,base_decline_rate,flagged_queue_p_at_20,lift_vs_tier_base
2,striking,7304,529,0.609529,1.00,0.390471
1,page_3_5,7242,528,0.561585,0.90,0.338415
0,page_1,11814,962,0.569663,0.55,-0.019663


### Segment Analysis Results
This table is calculated from the current run. `flagged_queue_p_at_20` uses only flagged candidates and caps the effective K at the number of flagged rows in that tier. `lift_vs_tier_base` compares that precision with the tier's overall decline rate. Small flagged counts make this estimate noisy; do not treat a high lift as conclusive evidence that the rule works better in that tier.


**Weak picks to check after running.**

- **Client concentration:** inspect the top-20 client counts printed in the concentration check. If a small number of clients dominate, consider a per-client cap or a log-scaled impressions term.
- **Tied staleness:** inspect the update-age counts in the top 20. If many rows share one update age, the ordering may mostly be driven by impressions rather than by degree of staleness.
- **Label mismatches:** compare the top rows' `trend_direction` values with `down`. A flagged row labelled `stable` or `up` is a mismatch against this evaluation label, though CTR underperformance could still be an early warning.

**Leakage check.** The score uses `days_since_last_update`, `impressions_90d`, `position_tier`, and `ctr`. The evaluation label is derived from `trend_direction` and is not used in `action_score`, `reason_code`, or `action_label`. Re-check this separation if you change the rule or add features.

## Self-check

Before submitting, verify each item after a fresh **Run All**:

- [ ] Every section is filled with reasoning and supporting code.
- [ ] The notebook runs top to bottom without errors.
- [ ] No client names, URLs, or private queries appear in the submitted outputs.
- [ ] Claims use careful wording: observed, measured, directional, decision-support.
- [ ] The generated CSV is saved under `work/outputs/baseline_action_score.csv`.
- [ ] Commit the notebook to your repository under `work/notebooks/` and submit the repository URL on the assignment card.